# Climate Data Download

Ask for climate data in plain language — *"daily rainfall and temperature for Honduras in 2020"* —
and get it on disk. A Gemini agent loads the
[`climate-data-download`](https://github.com/CGIAR-Climate-Data-Hub/skills) skill, routes each
variable to the right source, resolves your country or district to a bounding box, and this
notebook downloads it with `aggeodata` and stacks it into one analysis-ready NetCDF.

---

### Two Zarr decisions, both defaulted here

They share a word and are easy to confuse: **`use_zarr`** is how the data is *downloaded*,
**`output_format`** is how the finished cube is *written*. This notebook defaults both to Zarr.

#### 1. Downloading — `use_zarr: true`

The official endpoints hand out **one file per day** for CHIRPS and CHIRTS — throttled to a
single connection — and **queue one request per year** for AgERA5. A three-year request is
thousands of sequential HTTP round trips, or a morning in the CDS queue.

The same archives are published as consolidated **Zarr datacubes**, where a bounding box plus a
date range is one indexed read. This notebook sets `use_zarr: true` by default, which also lifts
the single-worker cap — the throttle belongs to `data.chc.ucsb.edu`, and the Zarr stores are a
different host entirely.

| Source | Store | Credentials |
|---|---|---|
| CHIRPS v3 daily | `aaguilar90/chirps-v3-daily-rnl` (Hugging Face) | none |
| CHIRTS-ERA5 daily | `aaguilar90/chirts-era5-daily` (Hugging Face) | none |
| AgERA5 | ECMWF ARCO datastore | the same `~/.cdsapirc` key |

The files written to disk are identical either way, so nothing downstream changes.

#### 2. Writing the cube — `output_format: "zarr"`, `zarr_codec: "zstd"`

The stacked cube is written as a **Zarr v3 store** laid out like the published archives — same
chunk geometry, codec and integer quantisation — so it can be sliced without reading the whole
thing. `zstd` is the codec GDAL, QGIS and R `terra` can actually open; the smaller `blosc`
default gives *"blosc compressor not available"* outside Python.

A store is not a NetCDF file, so read it back with `open_cube`, not `xr.open_dataset`:

```python
from aggeodata.transform.zarr_export import open_cube
ds = open_cube("climate_honduras_2020_2020.zarr")   # float32, CRS intact
```

**Need a `.nc` instead?** Say so — the agent will pass `output_format: "both"`, which writes the
store *and* a NetCDF. Or convert one you already have:
`open_cube("cube.zarr").to_netcdf("cube.nc")`.

> **When to turn it off.** Continuing a time series you already downloaded from UCSB (the mirror
> is CHIRPS v3 station-blended, the UCSB endpoint is CHIRP v3 satellite-only — *different
> products*); asking for the original CHIRTS-daily v1.0 (`chirts_source: chirts`) or monthly
> data, both rejected by the Zarr backend. Tell the agent and it will use the per-file endpoints.

### How to use this notebook

1. **Runtime ▸ Run all.** Setup takes about two minutes.
2. Edit the request in section **4** and answer the agent's questions. Pressing Enter on an empty
   box accepts what it proposes.
3. The maps and the finished cube appear in section **5**.

### Before you start

| Requirement | How |
|---|---|
| Gemini API key | Create one at [aistudio.google.com](https://aistudio.google.com/apikey), then add it in Colab under **Secrets** as `GOOGLE_API_KEY` with notebook access enabled. |
| Nothing else | CHIRPS, CHIRTS and NASA POWER need no account. **AgERA5 is the one exception** — it needs a free [CDS](https://cds.climate.copernicus.eu/) key in `~/.cdsapirc`. Without one, ask for solar radiation from NASA POWER instead (0.5°, no key). |


---
## 1 · Setup


In [ ]:
#@title Install aggeodata and the Gemini SDK { display-mode: "form" }
# The [download] extra carries cdsapi, s3fs and zarr — zarr is what the
# use_zarr backend reads, so it is required, not optional, for this notebook.
%pip install -q "aggeodata[download] @ git+https://github.com/CGIAR-Climate-Data-Hub/aggeodata.git"
%pip install -q -U "google-genai>=2.9.0"
print("Install finished.")

In [ ]:
#@title Environment check { display-mode: "form" }
import importlib.metadata as md_, importlib, pathlib

missing = []
for pkg in ("aggeodata", "zarr", "xarray"):
    try:
        print(f"OK       {pkg} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"MISSING  {pkg}")
        missing.append(pkg)
if missing:
    raise SystemExit(f"Missing: {', '.join(missing)}. Re-run the install cell; if it still "
                     "fails, use Runtime > Restart session and run all.")

# use_zarr needs a release that actually has the option.
from aggeodata.config.schemas import GeneralConfig
if not hasattr(GeneralConfig(), "use_zarr"):
    raise SystemExit("This aggeodata build predates use_zarr — reinstall from the repo HEAD.")
print(f"\nuse_zarr available (package default: {GeneralConfig().use_zarr}; this notebook uses True)")

OUT_DIR = pathlib.Path("/content/climate_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# AgERA5 is the only source needing credentials. Say so once, up front.
CDS_OK = pathlib.Path.home().joinpath(".cdsapirc").exists()
print(f"CDS key (~/.cdsapirc): {'found — AgERA5 available' if CDS_OK else 'not found — NASA POWER will be used for rsds'}")
print(f"output: {OUT_DIR}")

In [ ]:
#@title Connect to the Gemini API { display-mode: "form" }
SECRET_NAME = "GOOGLE_API_KEY" #@param {type:"string"}

import uuid, json
from google import genai
from google.colab import userdata
from IPython.display import display, Markdown
import ipywidgets as widgets

client = genai.Client(api_key=userdata.get(SECRET_NAME))
AGENT  = "antigravity-preview-05-2026"
print("Connected.")

---
## 2 · The download pipeline

Three functions, matching the skill's workflow: resolve the area, download, stack.

**`ncores` is derived, not asked.** The cap of 1 exists because `data.chc.ucsb.edu` bans more
than one connection — a rule about *that server*. When every CHIRPS/CHIRTS variable in the plan
reads from Zarr, nothing reaches it and workers fetch concurrent time-batches instead. One
per-file variable in the plan reinstates the cap for the whole run, so the notebook works it out
from the plan and reports which rule applied.


In [ ]:
#@title Pipeline - resolve extent, download, build the cube { display-mode: "form" }
import time, traceback
import yaml
import numpy as np
import xarray as xr

# Only these three have Zarr datacubes; every other source ignores use_zarr,
# and setting it on them is a validation error.
ZARR_SOURCES = {"chirps", "chirts", "agera5"}
# The default routing from the skill, as CF name -> source.
DEFAULT_SOURCE = {
    "pr": "chirps", "tasmax": "chirts", "tasmin": "chirts",
    "rsds": "nasa_power", "tas": "nasa_power", "hurs": "nasa_power", "sfcWind": "nasa_power",
    "vp": "agera5", "vpd": "agera5", "etr": "agera5", "tdps": "agera5",
    "hurs_06": "agera5", "hurs_09": "agera5", "hurs_12": "agera5",
    "hurs_15": "agera5", "hurs_18": "agera5",
}

state = {}


def open_cube_any(path):
    """Open a cube whatever format it is in.

    A Zarr store is not a NetCDF file: xr.open_dataset cannot read one, and the
    plain zarr reader leaves spatial_ref as a data variable (so .rio.crs returns
    None) and lets the CF decoder promote the quantised variables to float64.
    aggeodata's open_cube handles both, so use it for .zarr and keep
    xr.open_dataset for .nc.
    """
    path = pathlib.Path(path)
    if path.suffix == ".zarr":
        from aggeodata.transform.zarr_export import open_cube
        return open_cube(str(path))
    return xr.open_dataset(str(path))


def _say(msg):
    try:
        with out:
            print(msg)
    except NameError:
        print(msg)


def resolve_extent(country_code=None, feature_name=None, adm_level=1, bbox=None, **_):
    """Country or admin unit -> [xmin, ymin, xmax, ymax], the way aggeodata does it."""
    try:
        if bbox:
            return {"state": "ok", "bbox": [float(v) for v in bbox], "source": "given"}
        if not country_code:
            return {"state": "need_input", "note": "give a country_code (ISO3) or a bbox"}

        from aggeodata.ingestion.boundaries import get_admin_boundary, _fetch_geojson_cached
        if feature_name:
            gdf = get_admin_boundary(country_code, feature_name, adm_level)
        else:
            try:
                gdf = _fetch_geojson_cached(country_code, 0)
            except Exception:
                gdf = _fetch_geojson_cached(country_code, 1)
        xmin, ymin, xmax, ymax = (round(float(v), 4) for v in gdf.total_bounds.tolist())
        span = max(xmax - xmin, ymax - ymin)
        return {"state": "ok",
                "bbox": [xmin, ymin, xmax, ymax],
                "source": f"{country_code} {feature_name or '(whole country)'} adm{adm_level}",
                "span_deg": round(span, 3),
                "approx_chirps_pixels": int((xmax - xmin) / 0.05) * int((ymax - ymin) / 0.05)}
    except Exception as exc:
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "note": "If the unit name is wrong, call list_admin_units to see the spellings."}


def list_admin_units(country_code, adm_level=1, **_):
    """The exact spellings accepted by resolve_extent's feature_name."""
    try:
        from aggeodata.ingestion.boundaries import list_admin_units as _list
        names = _list(country_code, adm_level)
        return {"state": "ok", "count": len(names), "adm_level": adm_level,
                "units": names[:200],
                "truncated": len(names) > 200}
    except Exception as exc:
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}"}


def plan_ncores(var_cfg):
    """The skill's ncores rule, as code. Reads the RESOLVED per-variable blocks, so a
    single variable that fell back to the per-file backend reinstates the cap."""
    if any(b["source"] == "gee" for b in var_cfg.values()):
        return 1, "GEE present — its HDF5 writer is not parallel-safe"
    per_file = [v for v, b in var_cfg.items()
                if b["source"] in ("chirps", "chirts") and not b.get("use_zarr")]
    if per_file:
        return 1, (f"{', '.join(per_file)} read from data.chc.ucsb.edu, which bans "
                   ">1 connection")
    if any(b["source"] in ("chirps", "chirts") for b in var_cfg.values()):
        return 4, "every CHIRPS/CHIRTS variable reads from Zarr — the UCSB cap does not apply"
    return 4, "no UCSB per-file sources in the plan"


def _as_source_map(overrides):
    """[{variable, source}, ...] -> {variable: source}. Tolerates a plain dict too."""
    if not overrides:
        return {}
    if isinstance(overrides, dict):
        return dict(overrides)
    out_map = {}
    for item in overrides:
        if isinstance(item, dict) and item.get("variable") and item.get("source"):
            out_map[item["variable"]] = item["source"]
    return out_map


def run_download(variables, start_date, end_date, bbox, suffix="run",
                 use_zarr=True, source_overrides=None, sources=None, chirts_source="era5",
                 agera5_chunking="geo", build_cube=True, output_format="zarr",
                 download_format="zarr", **_):
    """Write the aggeodata config, download, and optionally stack into one cube."""
    try:
        return _run_download(variables, start_date, end_date, bbox, suffix, use_zarr,
                             _as_source_map(source_overrides or sources),
                             chirts_source, agera5_chunking, build_cube,
                             output_format, download_format)
    except Exception as exc:
        _say(f"\nDownload error: {exc}")
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc()[-1500:]}


def _run_download(variables, start_date, end_date, bbox, suffix, use_zarr,
                  sources, chirts_source, agera5_chunking, build_cube, output_format,
                  download_format):
    t0 = time.time()
    run_dir = OUT_DIR / suffix
    run_dir.mkdir(parents=True, exist_ok=True)
    # The yearly store names embed this file's STEM ({source}_{aoi}_{var}_{year}.zarr),
    # so the config is named after the area: hnd_2020.yaml -> chirps_hnd_2020_pr_2020.zarr.
    cfg_path = run_dir / f"{suffix}.yaml"

    # Per-variable blocks. use_zarr only goes on sources that have a Zarr store —
    # putting it on nasa_power or gee is a validation error, not a no-op.
    var_cfg, zarr_on, zarr_off, forced = {}, [], [], []
    for v in variables:
        src = sources.get(v, DEFAULT_SOURCE.get(v, "nasa_power"))
        block = {"source": src}
        if src in ZARR_SOURCES:
            v_zarr = bool(use_zarr)
            if src == "chirts":
                block["chirts_source"] = chirts_source
                # The mirror carries CHIRTS-ERA5 only. The config schema ACCEPTS
                # chirts_source: chirts alongside use_zarr, and the downloader then
                # raises partway through the run — so fall back here and say so.
                if chirts_source == "chirts" and v_zarr:
                    v_zarr = False
                    forced.append(v)
            block["use_zarr"] = v_zarr
            if src == "agera5" and v_zarr:
                block["agera5_chunking"] = agera5_chunking
            (zarr_on if v_zarr else zarr_off).append(v)
        var_cfg[v] = block

    ncores, why = plan_ncores(var_cfg)
    if forced:
        _say(f"note: {', '.join(forced)} fell back to the per-file backend — "
             "chirts_source: chirts has no Zarr mirror")
    ref = "pr" if "pr" in var_cfg else next(iter(var_cfg))

    cfg = {
        "dates": {"starting_date": start_date, "ending_date": end_date},
        "spatial_info": {"spatial_file": None, "extent": [float(v) for v in bbox]},
        "climate": {"variables": var_cfg},
        "soil": {"enabled": False},
        "general": {
            "suffix": suffix,
            "use_zarr": bool(use_zarr),     # default for every chirps/chirts/agera5 variable
            "ncores": ncores,
            "task": "download",
            "reference_variable": ref,
            "agera5_version": "2_0",
            "nasa_power_backend": "s3",
            "target_crs": "EPSG:4326",
            # One store per variable per year instead of one NetCDF per day. The
            # dailies are deleted only after each year's store is reopened and
            # verified, so an interrupted run never loses data.
            "download_format": download_format,
            # Zarr v3 + zstd: chunked and sliceable, and zstd is the codec GDAL,
            # QGIS and R terra can read (blosc gives "compressor not available").
            # "netcdf" or "both" when a .nc is wanted as well.
            "output_format": output_format,
            "zarr_codec": "zstd",
            "zarr_quantize": True,
            "zarr_sharded": False,      # GDAL rejects sharding_indexed
        },
        "paths": {"output_path": str(run_dir)},
    }
    cfg_path.write_text(yaml.dump(cfg, sort_keys=False), encoding="utf-8")

    # Report the RESOLVED backend, not the flag that was asked for — a variable may
    # have fallen back above, and saying "Zarr" while ncores is 1 reads as a bug.
    if zarr_on and zarr_off:
        backend = "mixed"
        banner = f"Zarr for {', '.join(zarr_on)}; per-file for {', '.join(zarr_off)}"
    elif zarr_on:
        backend, banner = "zarr", "Zarr datacubes"
    else:
        backend, banner = "per-file", "per-file endpoints"

    _say(f"config -> {cfg_path.name}")
    _say(f"backend: {banner}  | ncores {ncores} ({why})")
    _say(f"downloading {', '.join(variables)} for {start_date} .. {end_date} ...")

    _say(f"on disk: {'one Zarr store per variable per year' if download_format == 'zarr' else 'one NetCDF per day'}")

    from aggeodata.pipelines.download import run_download as _dl
    res = _dl(str(cfg_path))
    # With download_format=zarr each variable maps to {year: store_path}; with
    # netcdf it maps to a list of daily files. Count whichever came back.
    n_items = sum(len(v) for v in res.values()) if isinstance(res, dict) else 0
    stores = sorted(str(q) for q in run_dir.rglob("*.zarr") if q.is_dir())
    _say(f"  {n_items} item(s)" + (f", {len(stores)} yearly store(s)" if stores else ""))

    result = {
        "state": "ok",
        "backend": backend,
        "zarr_variables": zarr_on, "per_file_variables": zarr_off,
        "forced_to_per_file": forced,   # chirts_source: chirts has no Zarr mirror
        "ncores": ncores, "ncores_reason": why,
        "download_format": download_format,
        "items_downloaded": n_items,
        "yearly_stores": [pathlib.Path(q).name for q in stores],
        "config": str(cfg_path),
        "minutes": round((time.time() - t0) / 60, 1),
    }

    if build_cube:
        _say(f"stacking into one cube ({output_format}) ...")
        from aggeodata.pipelines.datacube import run_datacube
        # Returns the .zarr DIRECTORY for output_format="zarr", the .nc FILE otherwise.
        cube = pathlib.Path(run_datacube(str(cfg_path)))
        ds = open_cube_any(cube)
        result["cube"] = {
            "path": str(cube),
            "format": "zarr" if cube.suffix == ".zarr" else "netcdf",
            "variables": [v for v in ds.data_vars if v != "spatial_ref"],
            "dims": {k: int(v) for k, v in ds.sizes.items()},
            "is_zarr_store": cube.suffix == ".zarr",
        }
        ds.close()
        state.update(cube=cube, suffix=suffix, variables=list(variables))
        _say(f"  -> {cube.name}  {result['cube']['dims']}")
        if cube.suffix == ".zarr":
            _say("  note: a .zarr store — open it with open_cube(), not xr.open_dataset(). "
                 "Rebuild with output_format='both' if you also want a .nc.")

    result["minutes"] = round((time.time() - t0) / 60, 1)
    _say(f"\nfinished in {result['minutes']} min")
    return result


print("Pipeline loaded.")

In [ ]:
#@title Plotting - monthly maps and a domain-mean series { display-mode: "form" }
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "font.size": 9})

# How each CF variable should be summarised over a month, and how it is drawn.
HOW = {
    "pr":      ("sum",  "Blues",   "monthly rainfall (mm)"),
    "tasmax":  ("mean", "YlOrRd",  "mean daily Tmax"),
    "tasmin":  ("mean", "YlGnBu",  "mean daily Tmin"),
    "rsds":    ("mean", "inferno", "mean daily solar radiation"),
    "tas":     ("mean", "RdYlBu_r", "mean daily temperature"),
    "hurs":    ("mean", "BuPu",    "mean relative humidity"),
    "sfcWind": ("mean", "PuBuGn",  "mean wind speed"),
}


def _panels(n):
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.0 * rows), squeeze=False)
    return fig, [a for row in axes for a in row]


def plot_maps(variables=None, **_):
    """Monthly panels per variable, plus a domain-mean time series for each."""
    try:
        return _plot_maps(variables)
    except Exception as exc:
        import traceback as _tb
        _say(f"Plotting error: {exc}")
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "traceback": _tb.format_exc()[-1200:]}


def _plot_maps(variables=None):
    if not state.get("cube"):
        return {"state": "nothing_to_plot", "note": "no cube built yet"}
    ds = open_cube_any(state["cube"])
    tdim = next((d for d in ("time", "date") if d in ds.dims), None)
    if tdim is None:
        ds.close()
        return {"state": "no_time_axis", "note": f"dims are {list(ds.dims)}"}

    candidates = variables or [v for v in ds.data_vars if v != "spatial_ref"]
    wanted = [v for v in candidates if v in ds.data_vars]
    summary = {}
    for var in wanted:
        how, cmap, label = HOW.get(var, ("mean", "viridis", var))
        monthly = getattr(ds[var].resample({tdim: "1MS"}), how)()
        n = int(monthly.sizes[tdim])
        fig, axes = _panels(n)
        vals = monthly.values
        vmin = float(np.nanpercentile(vals, 2))
        vmax = float(np.nanpercentile(vals, 98))
        im = None
        for ax, t in zip(axes, monthly[tdim].values):
            im = monthly.sel({tdim: t}).plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                                             add_colorbar=False)
            ax.set_title(str(t)[:7]); ax.set_xlabel(""); ax.set_ylabel("")
        for ax in axes[n:]:
            ax.axis("off")
        if im is not None:
            fig.colorbar(im, ax=axes, shrink=0.7, label=label)
        fig.suptitle(f"{var} — {label} — {state['suffix']}", y=1.02)
        png = state["cube"].parent / f"{var}_monthly_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()

        series = ds[var].mean(dim=[d for d in ds[var].dims if d != tdim])
        fig2, ax2 = plt.subplots(figsize=(9, 2.4))
        series.plot(ax=ax2, lw=0.8)
        ax2.set_title(f"{var} — domain mean, daily"); ax2.set_xlabel("")
        plt.tight_layout(); plt.show()

        summary[var] = {
            "monthly": {str(t)[:7]: round(float(np.nanmean(monthly.sel({tdim: t}).values)), 2)
                        for t in monthly[tdim].values},
            "daily_mean": round(float(np.nanmean(ds[var].values)), 2),
            "daily_min": round(float(np.nanmin(ds[var].values)), 2),
            "daily_max": round(float(np.nanmax(ds[var].values)), 2),
            "png": str(png),
        }
    ds.close()
    return {"state": "ok", "variables": summary}


print("Plotting loaded.")

---
## 3 · Agent tools

Four functions run **here in the notebook**, not in the agent's sandbox — the sandbox has no
keyboard, no open internet and nowhere to put the files:

- **`ask_user`** — poses a batch of questions and returns every answer together.
- **`list_admin_units`** / **`resolve_extent`** — the exact spellings a country uses, and the
  bounding box for a country or one of its districts.
- **`run_download`** — writes the aggeodata config, downloads, and stacks the result.
- **`plot_maps`** — monthly panels and a domain-mean series per variable.

**Why answering is free.** The tools are *batched*: the agent sends a dozen questions in one
turn, you answer them one at a time locally, and the notebook spends a single API call sending
every answer back. Questions cost nothing — **turns** cost. The cell also backs off and retries
when the free tier returns a `429`.


In [ ]:
#@title Tool declarations { display-mode: "form" }
ask_user_tool = {
    "type": "function",
    "name": "ask_user",
    "description": (
        "Ask the user for a BATCH of parameters in a single call and receive every answer "
        "together. Put EVERY field you still need into `questions` - the client presents them "
        "one at a time and returns all the answers in one response, so a batch of 10 costs "
        "exactly what a batch of 1 costs. Calling this with one question and waiting burns the "
        "user's quota and is the most common way this workflow fails."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "description": "Every parameter you need right now. Longer is better.",
                "items": {
                    "type": "object",
                    "properties": {
                        "field":      {"type": "string", "description": "Parameter, e.g. 'end_date'."},
                        "question":   {"type": "string", "description": "The question, in plain language."},
                        "suggestion": {"type": "string", "description": "Draft value the user accepts with Enter."},
                    },
                    "required": ["field", "question"],
                },
            },
        },
        "required": ["questions"],
    },
}

list_admin_units_tool = {
    "type": "function",
    "name": "list_admin_units",
    "description": (
        "List the administrative unit names a country actually uses, at a given level "
        "(1 = region/province, 2 = district). Call this BEFORE resolve_extent whenever the user "
        "names a sub-national area, so you pass a spelling the boundary file contains rather "
        "than guessing. Runs in the notebook; your sandbox has no internet."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "country_code": {"type": "string", "description": "ISO 3166-1 alpha-3, e.g. HND."},
            "adm_level":    {"type": "integer", "description": "1 = region/province, 2 = district."},
        },
        "required": ["country_code"],
    },
}

resolve_extent_tool = {
    "type": "function",
    "name": "resolve_extent",
    "description": (
        "Turn a country, or one admin unit inside it, into [xmin, ymin, xmax, ymax]. Pass a "
        "bbox straight through if the user gave one. Runs in the notebook. Always show the user "
        "the bbox you intend to download before starting - never invent coordinates. The result "
        "reports the approximate CHIRPS pixel count, which is the honest measure of how big the "
        "request is."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "country_code": {"type": "string", "description": "ISO 3166-1 alpha-3, e.g. HND."},
            "feature_name": {"type": "string", "description": "One admin unit, spelled as list_admin_units returned it."},
            "adm_level":    {"type": "integer", "description": "Level the feature_name belongs to."},
            "bbox":         {"type": "array", "items": {"type": "number"},
                             "description": "[xmin, ymin, xmax, ymax] if the user gave one."},
        },
        "required": [],
    },
}

run_download_tool = {
    "type": "function",
    "name": "run_download",
    "description": (
        "Download the requested variables and stack them into one NetCDF. Runs in the notebook; "
        "your sandbox cannot reach the data. Call ONCE, after the user confirms the plan.\n"
        "use_zarr defaults to TRUE and should stay true: it reads the analysis-ready Zarr "
        "datacubes instead of one file per day (CHIRPS/CHIRTS) or one queued CDS request per "
        "year (AgERA5), and it lifts the single-worker cap. Set it false ONLY when the user is "
        "continuing a series already downloaded from the UCSB servers, wants the original "
        "CHIRTS-daily v1.0 (chirts_source: chirts), or wants monthly data - the Zarr backend "
        "rejects the last two outright.\n"
        "ncores is derived from the plan, not passed: the notebook applies the skill's rule and "
        "reports which case it hit."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "variables":  {"type": "array", "items": {"type": "string"},
                           "description": "CF names: pr, tasmax, tasmin, rsds, tas, hurs, sfcWind, vp, vpd, etr, tdps, hurs_06/09/12/15/18."},
            "start_date": {"type": "string", "description": "YYYY-MM-DD."},
            "end_date":   {"type": "string", "description": "YYYY-MM-DD."},
            "bbox":       {"type": "array", "items": {"type": "number"},
                           "description": "[xmin, ymin, xmax, ymax] from resolve_extent."},
            "suffix":     {"type": "string", "description": "Short label, no spaces, e.g. 'honduras_2020'."},
            "use_zarr":   {"type": "boolean", "description": "Default true. See the description above before setting false."},
            # A list of pairs, not a {variable: source} map: an object with arbitrary
            # keys has no `properties`, and the function-declaration schema rejects it.
            "source_overrides": {
                "type": "array",
                "description": "Optional per-variable source overrides, where the routing "
                               "default is not what you want (e.g. rsds from agera5).",
                "items": {
                    "type": "object",
                    "properties": {
                        "variable": {"type": "string", "description": "CF name, e.g. rsds."},
                        "source":   {"type": "string", "description": "chirps | chirts | agera5 | nasa_power | gee."},
                    },
                    "required": ["variable", "source"],
                },
            },
            "chirts_source":   {"type": "string", "description": "'era5' (default) or 'chirts' — the latter forces use_zarr false."},
            "agera5_chunking": {"type": "string", "description": "'geo' (default, many days over a modest area) or 'time'."},
            "build_cube": {"type": "boolean", "description": "Stack the per-day files into one cube. Default true."},
            "download_format": {"type": "string", "description":
                "What the download leaves on disk: 'zarr' (default — one store per variable "
                "per year, far fewer files, and re-running only fetches days the stores do "
                "not already cover) or 'netcdf' (one file per day). Use 'netcdf' only if the "
                "user wants the per-day files."},
            "output_format": {"type": "string", "description":
                "How the stacked cube is written: 'zarr' (default — Zarr v3 + zstd, chunked, "
                "sliceable, readable by GDAL/QGIS/R), 'netcdf', or 'both'. Keep the default "
                "unless the user says they need a NetCDF file, in which case pass 'both'. This "
                "is a different decision from use_zarr, which is about downloading."},
        },
        "required": ["variables", "start_date", "end_date", "bbox", "suffix"],
    },
}

plot_maps_tool = {
    "type": "function",
    "name": "plot_maps",
    "description": (
        "Draw monthly maps and a domain-mean daily series for each variable in the cube. Call "
        "once, after run_download succeeds. Returns the numbers behind every panel so you can "
        "describe them - you cannot see the images."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "variables": {"type": "array", "items": {"type": "string"},
                          "description": "Omit to plot every variable in the cube."},
        },
        "required": [],
    },
}

TOOLS = [ask_user_tool, list_admin_units_tool, resolve_extent_tool,
         run_download_tool, plot_maps_tool]
print("Tools declared:", [t["name"] for t in TOOLS])

In [ ]:
#@title Chat plumbing - the question box and the turn loop { display-mode: "form" }
import re

MAX_RETRIES = 4

out  = widgets.Output()
box  = widgets.Text(layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Send", button_style="primary")
ui   = widgets.HBox([box, send])

state.update({"result": None, "env": None, "queue": [], "answers": [],
              "subq": None, "collected": {}, "batch": 0, "turns": 0})

LOCAL = {"list_admin_units": list_admin_units, "resolve_extent": resolve_extent,
         "run_download": run_download, "plot_maps": plot_maps}


def api_call(**kwargs):
    """interactions.create with backoff, so a free-tier 429 pauses instead of crashing."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = client.interactions.create(**kwargs)
            state["turns"] += 1
            return res
        except Exception as exc:
            text = str(exc)
            if "429" not in text and "too_many_requests" not in text.lower():
                raise
            found = re.search(r"retry in ([\d.]+)s", text)
            wait  = float(found.group(1)) + 2 if found else 20 * attempt
            with out:
                print(f"rate limited - waiting {wait:.0f}s "
                      f"(attempt {attempt}/{MAX_RETRIES}; the notebook will resume on its own)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries - wait a minute and re-run, "
                       "or use a paid key.")


def show(obj, prefix="**Agent**\n\n"):
    text = obj if isinstance(obj, str) else getattr(obj, "output_text", None)
    if text:
        display(Markdown(prefix + text))


def pending_calls(res):
    done = {s.call_id for s in res.steps if s.type == "function_result"}
    return [s for s in res.steps if s.type == "function_call" and s.id not in done]


def call_args(call):
    return call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)


def reply(call, payload):
    state["answers"].append({"type": "function_result", "name": call.name,
                             "call_id": call.id, "result": payload})


def end(msg):
    box.disabled = send.disabled = True
    with out:
        display(Markdown(f"### {msg}"))


def run_local(call):
    """Execute a non-interactive tool call here in the notebook.

    Nothing here is allowed to raise. A tool that dies takes the whole run with it
    otherwise: the loop unwinds, the chat box stays disabled, and the traceback is
    all the user gets. Returning the error instead lets the agent read it, say what
    went wrong, and offer the user a next step.
    """
    fn = LOCAL.get(call.name)
    if fn is None:
        return {"error": f"no local handler for {call.name}"}
    with out:
        display(Markdown(f"*running `{call.name}`...*"))
    try:
        res = fn(**call_args(call))
    except Exception as exc:
        import traceback as _tb
        tb = _tb.format_exc()
        with out:
            display(Markdown(f"**`{call.name}` failed** — {type(exc).__name__}: {exc}"))
            print(tb[-1500:])
        return {"state": "error", "tool": call.name,
                "error": f"{type(exc).__name__}: {exc}", "traceback": tb[-1500:]}
    if not isinstance(res, dict):
        res = {"state": "ok", "result": res}
    with out:
        if call.name == "resolve_extent" and res.get("bbox"):
            display(Markdown(f"- bbox `{res['bbox']}` — {res.get('source', '')} "
                             f"(~{res.get('approx_chirps_pixels', '?')} CHIRPS pixels)"))
        elif call.name == "list_admin_units" and res.get("units"):
            display(Markdown(f"- {res['count']} units at adm{res['adm_level']}: "
                             + ", ".join(res["units"][:12])
                             + (" …" if res["count"] > 12 else "")))
        elif call.name == "run_download" and res.get("state") == "ok":
            display(Markdown(
                f"**Downloaded** via **{res['backend']}** — {res['items_downloaded']} item(s) in "
                f"{res['minutes']} min, ncores {res['ncores']} ({res['ncores_reason']})"
                + (f"; {len(res['yearly_stores'])} yearly Zarr store(s)"
                   if res.get("yearly_stores") else "")))
    return res


def questions_of(call):
    args  = call_args(call)
    items = args.get("questions")
    if isinstance(items, list) and items:
        return [q for q in items if isinstance(q, dict) and q.get("question")]
    if args.get("question"):
        return [{"field": args.get("field", "value"), "question": args["question"],
                 "suggestion": args.get("suggestion")}]
    return []


def pose(q):
    done    = state["batch"] - len(state["subq"]) + 1
    counter = f"`{done}/{state['batch']}` " if state["batch"] > 1 else ""
    text    = f"{counter}**{q.get('field', 'question')}** - {q['question']}"
    if q.get("suggestion"):
        text += (f"\n\n> *Proposed:* {q['suggestion']}\n>\n"
                 "> *Press Enter on an empty box to accept, or type a replacement.*")
    with out:
        display(Markdown(text))
    box.placeholder = "Enter to accept" if q.get("suggestion") else "Your answer..."
    box.disabled = send.disabled = False
    box.value = ""


def advance_turn():
    box.disabled = send.disabled = True
    with out:
        print(f"thinking... (turn {state['turns'] + 1}, sending {len(state['answers'])} result(s))")
    state["result"] = api_call(agent=climate_agent.id,
                               previous_interaction_id=state["result"].id,
                               environment=state["env"], input=state["answers"], tools=TOOLS)
    state["answers"] = []
    with out:
        show(state["result"])
    ask_next()


def ask_next():
    """Drive the queue: run local tools silently, pause only for questions."""
    while True:
        while state["queue"] and state["queue"][0].name != "ask_user":
            call = state["queue"].pop(0)
            reply(call, run_local(call))

        if state["queue"]:
            call = state["queue"][0]
            if state["subq"] is None:
                state["subq"]      = questions_of(call)
                state["collected"] = {}
                state["batch"]     = len(state["subq"])
                if state["batch"] > 1:
                    with out:
                        display(Markdown(f"*{state['batch']} questions in this batch - answering "
                                         "all of them costs a single API turn.*"))
            if state["subq"]:
                return pose(state["subq"][0])
            state["queue"].pop(0)
            reply(call, {"answers": state["collected"]})
            state["subq"] = None
            continue

        if state["answers"]:
            return advance_turn()

        res = state["result"]
        if res.status != "requires_action":
            return end(f"Finished in {state['turns']} API turns.")
        calls = pending_calls(res)
        if not calls:
            return end(f"Finished in {state['turns']} API turns - nothing left to answer.")
        state["queue"] = calls


def submit(_):
    # A raise inside a widget callback leaves the box disabled and the run dead
    # with no way back, so report and re-enable instead.
    try:
        _submit()
    except Exception as exc:
        import traceback as _tb
        with out:
            display(Markdown(f"**The run stopped** — {type(exc).__name__}: {exc}"))
            print(_tb.format_exc()[-1500:])
            display(Markdown("Re-run the `start(PROMPT)` cell to try again. "
                             "Downloaded files and any finished cube are still on disk."))
        box.disabled = send.disabled = False


def _submit():
    if not state["subq"] or box.disabled:
        return
    msg, box.value = box.value.strip(), ""
    q = state["subq"].pop(0)
    answer = msg or q.get("suggestion") or ""
    with out:
        display(Markdown(f"**You** - {answer}"))
    state["collected"][q.get("field", f"q{len(state['collected'])}")] = answer
    ask_next()


send.on_click(submit)
box.on_submit(submit)   # Enter key (deprecated in ipywidgets 8 - harmless warning)


def start(first_input):
    try:
        _start(first_input)
    except Exception as exc:
        import traceback as _tb
        display(Markdown(f"**Could not start** — {type(exc).__name__}: {exc}"))
        print(_tb.format_exc()[-2000:])
        display(Markdown(
            "If this names a tool or schema, the tool declarations cell is the place to look. "
            "If it is a 429, wait a minute and re-run this cell."))


def _start(first_input):
    out.clear_output()
    state.update(result=None, env=None, queue=[], answers=[], subq=None,
                 collected={}, batch=0, turns=0)
    state["result"] = api_call(agent=climate_agent.id, input=first_input,
                               environment="remote", tools=TOOLS)
    state["env"] = state["result"].environment_id
    display(out, ui)
    with out:
        show(state["result"])
    ask_next()


print("Helpers loaded.")

---
## 4 · The agent &nbsp; ✏️ **EDIT ME**

The standing brief, with the full `SKILL.md` appended so the agent never spends a turn reading a
file. Re-run the agent cell after editing it — the brief is baked in at creation time.


In [ ]:
#@title Build the standing brief { display-mode: "form" }
import urllib.request

_SKILL_URL = ("https://raw.githubusercontent.com/CGIAR-Climate-Data-Hub/skills/main/"
              ".agents/skills/climate-data-download/SKILL.md")
with urllib.request.urlopen(urllib.request.Request(
        _SKILL_URL, headers={"User-Agent": "cdh-climate-notebook/1.0"}), timeout=30) as _r:
    SKILL_TEXT = _r.read().decode("utf-8")

BRIEF = """
You are a climate data assistant for the CGIAR Climate Data Hub. You route each variable a user
asks for to the right source, resolve their area, and download it. The full
climate-data-download skill is appended below. Follow its routing table and its workflow, with
six differences that apply in this notebook.

1. THE NOTEBOOK DOWNLOADS, NOT YOU. Your sandbox has no internet and nowhere to put files.
   `run_download` writes the aggeodata YAML, runs the pipeline and stacks the result; it is the
   only way data lands on disk. Never write configs or shell commands yourself, and never ask
   the user to run anything.

2. ZARR IS THE DEFAULT AND SHOULD STAY THAT WAY. `use_zarr` defaults to true. Say so in the
   plan, in one line: the Zarr datacubes replace one-file-per-day (CHIRPS/CHIRTS) and the CDS
   queue (AgERA5) with a single indexed read, and they lift the one-worker cap. Set it false
   only in the three cases the skill lists - continuing a series already downloaded from UCSB,
   `chirts_source: chirts`, or monthly data - and say which one applies. Do not ask the user
   about the backend unless one of those cases is in play; propose it and move on.

3. THREE SETTINGS SHARE THE WORD "ZARR", AND ALL THREE DEFAULT TO IT. Keep all three; state them
   in the plan in one line each; do not interrogate the user about any of them.
     - use_zarr        - how the data is FETCHED (the analysis-ready archives).
     - download_format - what is left ON DISK: "zarr" writes one store per variable per year
       instead of one file per day, and re-running only fetches days the existing stores do not
       cover. Use "netcdf" only if the user asks for the per-day files.
     - output_format   - how the STACKED CUBE is written. Pass "both" when the user says they
       also want a NetCDF; "zarr" otherwise, read back with open_cube().

4. DO NOT PASS ncores. The notebook derives it from the plan and reports which rule applied.
   Mentioning the number in your summary is good; asking the user for it is not.

5. RESOLVE THE AREA, DO NOT INVENT IT. For a sub-national request call `list_admin_units`
   first, then `resolve_extent` with a spelling that list returned - boundary files disagree
   with common usage often enough that guessing wastes a turn. For a whole country call
   `resolve_extent` with the ISO3 code alone. Always show the bbox before downloading. If the
   user gave a bbox, pass it straight through.

6. YOU HAVE NO CHAT CHANNEL. The only way to ask anything is `ask_user`, which takes a LIST of
   questions. A question written as message text will never be answered, so never end a message
   with one. Batch everything: the user pays per turn and every turn resends this conversation.
   Your first ask_user call should carry every parameter you still need. Aim to finish in under
   six turns.

PARAMETERS

  Propose, then confirm: the variable list as CF names, the date range, the suffix (a short slug
  like honduras_2020), and the source for each variable from the skill's routing table.

  Never guess: the area, and whether the user has a CDS key when AgERA5 is involved.

  AgERA5 is the only source needing credentials. The setup cell printed whether ~/.cdsapirc
  exists. If it does not and the user wants solar radiation, humidity or wind, route those to
  nasa_power (0.5 degree, no key) and say that is what you did and why. Reserve agera5 for the
  variables only it provides - vp, vpd, etr, tdps, hourly RH - and tell the user they need a key
  for those.

  CF names: pr, tasmax, tasmin, tas, rsds, hurs, sfcWind, vp, vpd, etr, tdps,
  hurs_06/09/12/15/18. Translate the user's words into these; never invent a name.

AFTER THE RUN

  Report what landed: the download backend, how many items and yearly stores, the elapsed
  minutes, the cube's
  format and path, and its variables and dimensions. When the cube is a .zarr store, say in one
  line that it opens with `open_cube()` rather than `xr.open_dataset()`. Then call `plot_maps` once. You cannot see the images, so describe
  the numbers it returns - the wettest and driest months, the temperature range, anything that
  looks wrong for the region and season. Close with where the files are and one honest caveat
  about the data itself (CHIRPS is satellite-and-station rainfall, not gauge truth; NASA POWER
  radiation is a 0.5 degree cell, so it is near-uniform over a small area).

--- FULL SKILL BELOW ---
"""

SYSTEM_INSTRUCTION = BRIEF + "\n" + SKILL_TEXT
print(f"System instruction: {len(SYSTEM_INSTRUCTION):,} chars "
      f"({len(BRIEF):,} brief + {len(SKILL_TEXT):,} skill)")

In [ ]:
#@title Create the agent { display-mode: "form" }
climate_agent = client.agents.create(
    id=f"climate-data-download-{uuid.uuid4().hex[:8]}",
    base_agent=AGENT,
    system_instruction=SYSTEM_INSTRUCTION,
    base_environment={"type": "remote"},
)
print(f"Agent ready: {climate_agent.id}")

### Your request &nbsp; ✏️ **EDIT ME**

A place, some variables and a period is enough. The agent fills in the sources and asks about
anything it should not assume.

| You want | Write something like |
|---|---|
| A country, one year | `daily rainfall and temperature for Honduras in 2020` |
| One district | `rainfall for Yoro department, Honduras, 2018 to 2020` |
| A bbox | `rainfall and solar radiation for bbox [-87.8, 14.0, -86.8, 15.0] in 2021` |
| Everything a crop model needs | `pr, tasmax, tasmin and rsds for Malawi, 2020` |


In [ ]:
PROMPT = "Daily rainfall and temperature for Honduras in 2020" #@param {type:"string"}

start(PROMPT)

---
## 5 · The maps again

The agent draws these during the conversation. Run this to redraw or export them without
spending another API turn.


In [ ]:
#@title Redraw and list the outputs { display-mode: "form" }
if state.get("cube"):
    summary = plot_maps()
    for var, s in summary.get("variables", {}).items():
        print(f"  {var:8} daily mean {s['daily_mean']:>8}  "
              f"range {s['daily_min']} .. {s['daily_max']}")
    print(f"\ncube: {state['cube']}")
    for p in sorted(state["cube"].parent.glob("*.png")):
        print(f"  figure: {p.name}")
else:
    print("No download in this session yet - run the cells above first.")

---
## What next?

- **Feed it to a crop model.** The
  [`spatial-crop-modeler` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/spatial_crop_modeler_example.ipynb)
  takes a weather cube and runs DSSAT over it. Check its own format requirements before you go —
  it documents what it needs, including which file formats it accepts.
- **Need a NetCDF rather than a store?** Either ask for `output_format: "both"` up front, or
  convert: `open_cube("cube.zarr").to_netcdf("cube.nc")`.
- **Document it** with the
  [`cdh-metadata` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/CDH_metadata_example.ipynb),
  which turns the cube into a CDH catalogue record.
- **Download the files before the runtime recycles.** Colab discards `/content` when the session
  ends. Mount Drive and copy `/content/climate_data` across if you want to keep it.
- **Mind the product you picked.** The Zarr mirror is CHIRPS v3 (station-blended); the UCSB
  endpoint is CHIRP v3 (satellite-only). Don't mix them inside one time series.
- **Read the skill**: [climate-data-download](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/climate-data-download/SKILL.md)
  · [aggeodata](https://github.com/CGIAR-Climate-Data-Hub/aggeodata)
- **More skills**: [CGIAR-Climate-Data-Hub/skills](https://github.com/CGIAR-Climate-Data-Hub/skills)
